In [22]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import os

In [23]:
raw_ihs_poverty_2010 = pd.read_csv(r'd:\GG\source\householdpoverty_10.CSV')
raw_ihs_weight_2010 = pd.read_csv(r'd:\GG\source\householdweight_10.CSV')
raw_ihs_weight_2015 = pd.read_csv(r'd:\GG\source\householdweight_15.CSV')
raw_ihs_poverty_2015 = pd.read_csv(r'd:\GG\source\householdpoverty_15.CSV')
raw_dhs_2020 = pd.read_stata(r'd:\GG\source\household_19_20.DTA')
raw_findex_2021 = pd.read_csv(r'd:\GG\source\connectivity_21.csv')
raw_findex_2024 = pd.read_csv(r'd:\GG\source\connectivity_24.csv')

In [24]:
# aggregate economic rank
from numpy import nan as NA

# 2010
df_2010 = pd.DataFrame()
df_2010['hid'] = raw_ihs_poverty_2010['hid']
df_2010['survey_year'] = 2010
df_2010['data_source'] = 'IHS'
df_2010['lga'] = raw_ihs_poverty_2010['lga']

weight_lookup = raw_ihs_weight_2010[['hid', 'weightslga']].drop_duplicates(subset=['hid'])
df_2010 = df_2010.merge(weight_lookup, on='hid', how='left')
df_2010['weightslga'] = df_2010['weightslga'].fillna(np.nan)

poverty_map = raw_ihs_poverty_2010.drop_duplicates('hid').set_index('hid')['s11q2']
df_2010['hh_income'] = df_2010['hid'].astype(int).map(poverty_map)

# 2015
df_2015 = pd.DataFrame()
df_2015['hh_id'] = raw_ihs_poverty_2015['hid']
df_2015['survey_year'] = 2015
df_2015['data_source'] = 'IHS'
df_2015['eanum'] = raw_ihs_poverty_2015['eanum']

weight_lookup = raw_ihs_weight_2015[['eanum', 'hhweight']].drop_duplicates(subset=['eanum'])
df_2015 = df_2015.merge(weight_lookup, on='eanum', how='left')
df_2015['hh_weight'] = df_2015['hhweight'].fillna(np.nan)

poverty_map = raw_ihs_poverty_2015.drop_duplicates('hid').set_index('hid')['s13q3']
df_2015['hh_income'] = df_2015['hh_id'].astype(int).map(poverty_map)

# 2020
df_2020 = pd.DataFrame()
df_2020['hh_id'] = raw_dhs_2020['hhid']
df_2020['survey_year'] = raw_dhs_2020['hv007']
df_2020['hh_income'] = raw_dhs_2020['hv270']

df_2020['data_source'] = 'DHS'
df_2020['weight'] = raw_dhs_2020['hv005']

quintile_labels = {'poorest': 1, 'poorer': 2, 'middle': 3, 'richer': 4, 'richest': 5}
df_2020['hh_income'] = df_2020['hh_income'].map(quintile_labels)

# 2021
df_2021 = pd.DataFrame()
df_2021['hh_id'] = raw_findex_2021.index.map(lambda x: f"findex_21_{x}")
df_2021['survey_year'] = 2021
df_2021['data_source'] = 'Findex'
df_2021['hh_income'] = raw_findex_2021['inc_q']
df_2021['weight'] = (raw_findex_2021['wgt']).fillna(NA)

# 2024
df_2024 = pd.DataFrame()
df_2024['hh_id'] = raw_findex_2024.index.map(lambda x: f"findex_24_{x}")
df_2024['survey_year'] = 2024
df_2024['data_source'] = 'Findex'
df_2024['hh_income'] = raw_findex_2024['inc_q']
df_2024['weight'] = (raw_findex_2024['wgt']).fillna(NA)

In [25]:
# econ rank construct
df_2010['hh_econ_rank'] = (df_2010.sort_values('hh_income')['weightslga'].cumsum() - 0.5 * df_2010['weightslga']) / df_2010['weightslga'].sum() * 100
df_2015['hh_econ_rank'] = (df_2015.sort_values('hh_income')['hh_weight'].cumsum() - 0.5 * df_2015['hh_weight']) / df_2015['hh_weight'].sum() * 100
df_2020['hh_econ_rank'] = (df_2020.sort_values('hh_income')['weight'].cumsum() - 0.5 * df_2020['weight']) / df_2020['weight'].sum() * 100
df_2021['hh_econ_rank'] = (df_2021.sort_values('hh_income')['weight'].cumsum() - 0.5 * df_2021['weight']) / df_2021['weight'].sum() * 100
df_2024['hh_econ_rank'] = (df_2024.sort_values('hh_income')['weight'].cumsum() - 0.5 * df_2024['weight']) / df_2024['weight'].sum() * 100

In [26]:
# concat all
df_2010.rename(columns={'hid': 'hh_id', 'weightslga': 'weight'}, inplace=True)
df_2015.rename(columns={'hh_weight': 'weight'}, inplace=True)

target_cols = ['hh_id', 'survey_year', 'data_source', 'hh_income', 'weight', 'hh_econ_rank']

df_2010 = df_2010[target_cols]
df_2015 = df_2015[target_cols]
df_2020 = df_2020[target_cols]
df_2021 = df_2021[target_cols]
df_2024 = df_2024[target_cols]

dfs = [df_2010, df_2015, df_2020, df_2021, df_2024]
for df in dfs:
    df['hh_id'] = df['hh_id'].astype(str)

df_panel = pd.concat(dfs, ignore_index=True)
df_panel = df_panel.dropna(subset=['hh_id', 'hh_income', 'weight'])

df_panel['unique_id'] = (
    df_panel['data_source'] + '_' + 
    df_panel['survey_year'].astype(str) + '_' + 
    df_panel['hh_id']
)

df_panel.set_index('unique_id', inplace=True)

display(df_panel.tail())

,hh_id,survey_year,data_source,hh_income,weight,hh_econ_rank
unique_id,,,,,,
Findex_2024_findex_24_1003,findex_24_1003,2024,Findex,5.0,0.328288,81.159517
Findex_2024_findex_24_1004,findex_24_1004,2024,Findex,2.0,0.777492,20.392583
Findex_2024_findex_24_1005,findex_24_1005,2024,Findex,1.0,0.671328,19.898279
Findex_2024_findex_24_1006,findex_24_1006,2024,Findex,4.0,0.483620,79.927619
Findex_2024_findex_24_1007,findex_24_1007,2024,Findex,5.0,0.798636,99.960385


In [27]:
# CPI data
data = {
    "year": [
        2005, 2006, 2007, 2008, 2009,
        2010, 2011, 2012, 2013, 2014,
        2015, 2016, 2017, 2018, 2019,
        2020, 2021, 2022, 2023, 2024
    ],
    "CPI_All_Items": [
        45.38, 46.31, 48.80, 50.97, 53.29,
        55.98, 58.67, 61.16, 64.65, 68.49,
        73.16, 78.44, 84.75, 90.27, 96.70,
        102.43, 109.98, 122.64, 143.46, 160.05
    ]
}

cpi_gambia = pd.DataFrame(data)

In [28]:
# Economic Rank: Normalize and Extract Average 

from scipy.stats import norm
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Transform the Rank to Continuous Space
df_panel['rank_pct'] = df_panel['hh_econ_rank'].clip(0.1, 99.9) / 100
df_panel['rank_continuous'] = norm.ppf(df_panel['rank_pct'])

# Compute Weighted Average per Year
def weighted_mean(group):
    return np.average(group['rank_continuous'], weights=group['weight'])

df_cohort = df_panel.groupby('survey_year').apply(weighted_mean).reset_index(name='avg_rank_continuous')

# Create Complete Timeline (2005-2024) and Merge CPI
df_timeline = pd.DataFrame({'survey_year': range(2005, 2025)})
df_timeline = df_timeline.merge(df_cohort, on='survey_year', how='left')

cpi_gambia['cpi_growth'] = cpi_gambia['CPI_All_Items'].pct_change().fillna(0)
df_timeline = df_timeline.merge(cpi_gambia.rename(columns={'year': 'survey_year'}), on='survey_year', how='left')

In [29]:
# Fit State-Space Model
# Fit State-Space Model (AR(1) with Exogenous CPI Driver)
# Statsmodels automatically skips updates for missing (NaN) gap years
endog = df_timeline['avg_rank_continuous']
exog = df_timeline['cpi_growth']

model = SARIMAX(endog, exog=exog, order=(1, 0, 0), trend='c')
model_fit = model.fit(disp=False)

# Extract Predictions and Impute Pseudo Data
# The out-of-sample/missing periods use the transition equation automatically
df_timeline['imputed_continuous'] = model_fit.fittedvalues

# Inverse Transform back to 0-100 Rank Space
df_timeline['hh_econ_rank'] = norm.cdf(df_timeline['imputed_continuous']) * 100

# Finalize Pseudo Panel Format
df_timeline['data_source'] = df_timeline['avg_rank_continuous'].isna().map({True: 'Pseudo', False: 'Survey'})
df_pseudo_panel = df_timeline[['survey_year', 'data_source', 'CPI_All_Items', 'cpi_growth', 'hh_econ_rank']].copy()

display(df_pseudo_panel)

,survey_year,data_source,CPI_All_Items,cpi_growth,hh_econ_rank
0,2005,Pseudo,45.38,0.000000,47.469202
1,2006,Pseudo,46.31,0.020494,48.482171
2,2007,Pseudo,48.80,0.053768,50.128678
3,2008,Pseudo,50.97,0.044467,49.668348
4,2009,Pseudo,53.29,0.045517,49.720303
5,2010,Survey,55.98,0.050479,49.965865
6,2011,Pseudo,58.67,0.048053,52.530549
7,2012,Pseudo,61.16,0.042441,48.329569
8,2013,Pseudo,64.65,0.057063,50.863115
9,2014,Pseudo,68.49,0.059397,50.143732
